# Assignment 10

In [154]:
import numpy as np
import cv2
import mediapipe as mp
import pandas as pd
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [168]:

# def euclidean(x1, y1, z1, x2, y2, z2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2 + (z1-z2)**2)

# def euclidean(x1, y1, x2, y2):
#     return np.sqrt((x1-x2)**2 + (y1-y2)**2)

def euclidean(x1, x2):
    return np.abs((x1-x2))

video_tags = {"A":159, "B":22}
base_dir_mediapipe = "../../MainProject/Assignment10/data/csv_of_all_videos/"
base_dir_kinect = "../../MainProject/Assignment9/data/kinect_good_preprocessed/"

differences = []

for letter, number in video_tags.items():
    for i in range(1, number + 1):
        mediapipe_dir = base_dir_mediapipe + f"{letter}{i}_mediapipe.csv"
        kinect_dir = base_dir_kinect + f"{letter}{i}_kinect.csv"

        df_mediapipe = pd.read_csv(mediapipe_dir)
        df_kinect = pd.read_csv(kinect_dir)

        # Kinect files have a column named " head_x" instead of "head_x" for some reason...
        df_kinect = df_kinect.rename(columns={" head_x":"head_x"})

        # Only keep frames that exist in both mediapipe and kinect DataFrames
        frames = df_kinect["FrameNo"].values
        df_mediapipe = df_mediapipe.loc[df_mediapipe.index.isin(frames)]
        df_mediapipe = df_mediapipe.reset_index(drop=True)

        # Need this since mediapipe sets origin in upper left corner
        for column in [c for c in df_kinect.columns if c.endswith("_y")]:
            df_mediapipe[column] = 1 - df_mediapipe[column]

        # Set origin at the midpoint of the hips for both datasets
        for axis in ("_x", "_y", "_z"):

            # Setup kinect
            left_hip_kinect = df_kinect[f"left_hip{axis}"]
            right_hip_kinect = df_kinect[f"right_hip{axis}"]
            hip_mid_kinect = (left_hip_kinect + right_hip_kinect) / 2

            # Setup mediapipe
            left_hip_mediapipe = df_mediapipe[f"left_hip{axis}"]
            right_hip_mediapipe = df_mediapipe[f"right_hip{axis}"]
            hip_mid_mediapipe = (left_hip_mediapipe + right_hip_mediapipe) / 2

            right_hand_kinect = df_kinect[f"right_hand{axis}"]
            right_hand_mediapipe = df_mediapipe[f"right_hand{axis}"]
            scaling_factor = right_hand_kinect / right_hand_mediapipe

            axis_cols = [c for c in df_kinect.columns if c.endswith(axis)]
            for col in axis_cols:
                df_kinect[col] -= hip_mid_kinect
                df_mediapipe[col] -= hip_mid_mediapipe
                df_mediapipe[col] *= scaling_factor

        df_kinect = df_kinect.drop("FrameNo", axis=1)
        df_mediapipe = df_mediapipe.drop("FrameNo", axis=1)

        df_difference = np.abs(df_kinect - df_mediapipe)
        differences += [df_difference]

        df_percentages = np.abs(df_difference / df_kinect)*100
        break
    break

print(f"{'Node':<25} {'Kinect (m)':>10} {'MediaPipe (m)':>15} {'Error (m)':>12} {'Error (%)':>10}")
print("-" * 75)
for column in df_difference.columns:
    print(f"{column:<25} {df_kinect[column][0]:>10.3f} {df_mediapipe[column][0]:>15.3f} {df_difference[column][0]:>12.3f} {df_percentages[column][0]:>10.2f}")


Node                      Kinect (m)   MediaPipe (m)    Error (m)  Error (%)
---------------------------------------------------------------------------
head_x                         0.012           0.001        0.011      88.92
head_y                         0.720           0.340        0.379      52.72
head_z                         0.063          -0.018        0.080     127.99
left_shoulder_x               -0.139           0.031        0.170     122.41
left_shoulder_y                0.508           0.263        0.244      48.08
left_shoulder_z                0.063          -0.007        0.070     110.85
left_elbow_x                  -0.239           0.054        0.293     122.58
left_elbow_y                   0.732           0.360        0.372      50.85
left_elbow_z                   0.043          -0.011        0.054     125.66
right_shoulder_x               0.163          -0.024        0.187     114.69
right_shoulder_y               0.481           0.264        0.217      45.15
